In [12]:
import requests
import pandas as pd 

from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
BASE_URL = "https://openlibrary.org/search.json"

In [13]:
def create_session():
    retry_strategy = Retry(
        total = 3,
        connect = 3,
        read = 3,
        backoff_factor = 2,
        status_forcelist = [429,500,502,503,504],
        allowed_methods=["GET"]
    )

    adapter = HTTPAdapter(
        max_retries = retry_strategy
    )

    session = requests.Session()

    session.mount(
        "https://",
        adapter
    )

    session.headers.update({
            "User-Agent": "BookVerse/0.1"
    })

    return session

session = create_session()

def search_openlibrary(query , limit = 100):

    params = {
        "q": query ,
        "limit":limit
    }

    try:

        response = requests.get(
            BASE_URL,
            params = params,
            timeout = 300
        )

        response.raise_for_status()

        return response.json()

    except requests.expections.Timeout:

        print(f"Request timed out for query: {query}")

        return None

    except requests.exceptions.RequestException as e:

        print(f"Request failed for '{query}' : {e}")

        return None

In [15]:
genres = ["fantasy", "horror", "romance", "thriller", "crime", "manga"]
books_per_genre = 10000

genre_dataframes = []

for genre in genres:
    response = session.get(
        BASE_URL,
        params={"q": genre, "limit": books_per_genre},
        timeout=(10, 120)
    )
    response.raise_for_status()

    data = response.json()
    genre_df = pd.DataFrame(data.get("docs", []))
    genre_df["genre"] = genre

    genre_dataframes.append(genre_df)
    print(f"{genre}: {len(genre_df)} books")

combined_df = pd.concat(genre_dataframes, ignore_index=True)

combined_df.to_csv("openlibrary_books.csv", index=False)

print(f"Combined dataframe shape: {combined_df.shape}")
combined_df.head()

fantasy: 10000 books
horror: 10000 books
romance: 10000 books
thriller: 10000 books
crime: 10000 books
manga: 10000 books
Combined dataframe shape: (60000, 25)


,author_key,author_name,cover_edition_key,cover_i,ebook_access,edition_count,first_publish_year,has_fulltext,ia,ia_collection,...,lending_identifier_s,series_key,series_name,series_position,subtitle,id_standard_ebooks,id_project_gutenberg,id_wikisource,id_librivox,genre
0,[OL666139A],[Carole Mortimer],OL10738221M,8337298.0,printdisabled,6,1983.0,True,"[fantasygirl0000mort_s1b6, fantasygirl0000mort]","[internetarchivebooks, openlibrary-d-ol, print...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,fantasy
1,[OL661093A],[Penny Jordan],OL10739155M,6611725.0,borrowable,4,1996.0,True,"[herchristmasfant0000jord, herchristmasfant00j...","[americana, inlibrary, internetarchivebooks, p...",...,herchristmasfant0000jord,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,fantasy
2,[OL451969A],[Emma Darcy],OL10738383M,5073358.0,borrowable,1,1985.0,True,[fantasy00darc],"[americana, inlibrary, internetarchivebooks, o...",...,fantasy00darc,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,fantasy
3,[OL59362A],[Nancy Friday],OL11758368M,2787375.0,borrowable,32,1973.0,True,"[diepegrondenerot0000frid, mysecretgardenwo000...","[americana, inlibrary, internetarchivebooks, p...",...,diepegrondenerot0000frid,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,fantasy
4,"[OL18977A, OL7301162A]","[Nora Roberts, Susan Ericksen]",NaN,6305245.0,printdisabled,20,2010.0,True,"[fantasyindeath00robb_0, fantaisieducrime0000r...","[americana, delawarecountydistrictlibrary, int...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,fantasy


In [17]:
# Shuffle all genres together and reset the index
shuffled_df = combined_df.sample(frac=1, random_state=42).reset_index(drop=True)

# Save the shuffled dataframe as a CSV file
shuffled_df.to_csv("books.csv", index=False)

# Import the saved CSV file
shuffled_books_df = pd.read_csv("shuffled_books.csv")

print(f"Shuffled dataframe shape: {shuffled_books_df.shape}")
shuffled_books_df.head()

Shuffled dataframe shape: (60000, 25)


,author_key,author_name,cover_edition_key,cover_i,ebook_access,edition_count,first_publish_year,has_fulltext,ia,ia_collection,...,lending_identifier_s,series_key,series_name,series_position,subtitle,id_standard_ebooks,id_project_gutenberg,id_wikisource,id_librivox,genre
0,"['OL7869817A', 'OL12644385A', 'OL7944520A']","['Emilia Dziubak', 'Madlena Szeliga', 'Karolin...",OL47348442M,13938293.0,no_ebook,1,2021.0,False,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,horror
1,['OL7683932A'],['J. Conrad'],NaN,NaN,no_ebook,1,2018.0,False,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,thriller
2,['OL4314449A'],['Karen Perry'],NaN,NaN,no_ebook,1,2021.0,False,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,thriller
3,['OL10911944A'],['Sally SINS'],NaN,NaN,no_ebook,2,2017.0,False,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,fantasy
4,['OL14445174A'],['Jeremiah Hoppe'],NaN,NaN,no_ebook,1,2017.0,False,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,fantasy
